> PostgreSQL 14+ practice. Run `sql/00_sample_data.sql` first, then `SET search_path TO sql_practice, public;` in this connection. Run cells in order in the same SQL session. See README.md for setup and expected results. These are SQL cells; the original Python notebook metadata did not make them executable as Python.


## Using the native PIVOT operator

In [ ]:
-- PostgreSQL FILTER replaces SQL Server PIVOT: turn status values into count columns.
SELECT p.provider_name AS provider,
    COUNT(*) FILTER (WHERE t.status = 'Success') AS successful_transactions,
    COUNT(*) FILTER (WHERE t.status = 'Failed') AS failed_transactions,
    COUNT(*) FILTER (WHERE t.status = 'Pending') AS pending_transactions
FROM payment_transactions t
JOIN payment_providers p ON p.provider_id = t.provider_id
GROUP BY p.provider_name
ORDER BY provider;

In [ ]:
-- Aggregate repeated product/month rows; missing months produce NULL, not zero.
SELECT product,
    SUM(sales) FILTER (WHERE month = 'Jan') AS jan,
    SUM(sales) FILTER (WHERE month = 'Feb') AS feb,
    SUM(sales) FILTER (WHERE month = 'Mar') AS mar
FROM sales
GROUP BY product
ORDER BY product;

## Pivot monthly successful payment amounts

### Business requirement: The finance team wants to compare monthly successful payment amounts for each payment provider.

In [ ]:
-- Conditional sums reshape Q1 successful amounts into one row per provider ID.
SELECT provider_id,
    SUM(CASE WHEN EXTRACT(MONTH FROM transaction_date) = 1 THEN amount ELSE 0 END) AS january_amount,
    SUM(CASE WHEN EXTRACT(MONTH FROM transaction_date) = 2 THEN amount ELSE 0 END) AS february_amount,
    SUM(CASE WHEN EXTRACT(MONTH FROM transaction_date) = 3 THEN amount ELSE 0 END) AS march_amount
FROM payment_transactions
WHERE status = 'Success'
    AND transaction_date >= DATE '2026-01-01'
    AND transaction_date < DATE '2026-04-01'
GROUP BY provider_id
ORDER BY provider_id;

## UNPIVOT in SQL

In [ ]:
-- LATERAL VALUES turns month columns into rows; filter NULLs to match SQL Server UNPIVOT behavior.
SELECT s.product, v.month, v.sales
FROM sales_pivoted s
CROSS JOIN LATERAL (VALUES
    ('Jan', s.jan), ('Feb', s.feb), ('Mar', s.mar)
) AS v(month, sales)
WHERE v.sales IS NOT NULL
ORDER BY s.product, v.month;